# Fink/LSST — Angular Uniformity of DIA Alerts in DDFs

This notebook tests the **angular uniformity** of DIA alert positions within each
LSST Deep Drilling Field (DDF) using the **two-point angular correlation function** $w(\theta)$.

Data are loaded from parquet files produced by notebook `01c_fink_dipoles_per_ddf.ipynb`
(directory `data_DIPOLES_01c/`).  **No Fink API calls are made here.**

## Scientific goals

1. **Auto-correlation of all alerts** (per DDF):  
   Is the spatial distribution of DIA detections consistent with a Poisson random field,
   or are there angular clustering signatures (bad columns, bright-star halos, …)?

2. **Auto-correlation of dipole alerts** (`isDipole=True`, per DDF):  
   Are dipoles clustered in specific sky regions?
   A significant positive $w(\theta)$ at small scales would indicate that dipoles
   are spatially concentrated (e.g. around bright stars or bad detectors).

3. **Cross-correlation non-dipoles × dipoles** (per DDF):  
   Do dipole and non-dipole alerts trace the same spatial field, or do dipoles
   preferentially appear where there are fewer/more non-dipole sources?

## Method

We use the **Landy–Szalay estimator** (1993):

$$w(\theta) = \frac{DD(\theta) - 2\,DR(\theta) + RR(\theta)}{RR(\theta)}$$

where $D$ is the data catalogue and $R$ is a random catalogue of the same size,
drawn uniformly inside the cone of radius `CONE_RADIUS` around each DDF centre.

We use **TreeCorr** when available; otherwise fall back to a pure-NumPy estimator
based on a KD-tree approach (slower but always available).

A flat $w(\theta) \approx 0$ indicates uniform (Poisson) distribution.
Positive values indicate clustering; negative values indicate anti-clustering (inhibition).

## Notes

- The angular scale probed is limited by the cone radius (~1 degree).
- For small samples ($N < 50$ dipoles) the correlation function is very noisy;
  the plots will clearly show the Poisson noise floor.
- This is an **exploratory analysis**: the goal is to decide whether a more rigorous
  treatment (mask, completeness correction, photo-z weighting) is warranted.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-23
- last update : 2026-05-23

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

# Try to import TreeCorr (optional but recommended)
try:
    import treecorr

    HAS_TREECORR = True
    print(f"treecorr version : {treecorr.__version__}  ✓")
except ImportError:
    HAS_TREECORR = False
    print("treecorr NOT found — using NumPy KD-tree fallback.")
    print("Install with:  pip install treecorr")

In [ ]:
# Enable interactive matplotlib backend
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Input: parquet files from notebook 01c ────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"  # read from here

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "DIPOLES_02"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input data : {os.path.abspath(DIR_DATA_IN)}")
print(f"Output data: {os.path.abspath(DIR_DATA)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

# ── DDF definitions (must match 01c) ─────────────────────────────────────────
CONE_RADIUS_DEG = 1.0  # cone radius used in 01c (3600 arcsec = 1 deg)

DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Correlation function binning ──────────────────────────────────────────────
THETA_MIN_DEG = 0.001  # minimum angular separation (degrees)
THETA_MAX_DEG = 0.9  # maximum angular separation (degrees)
N_THETA_BINS = 20  # number of log-spaced bins
N_RANDOM = 5000  # size of random catalogue per DDF
RANDOM_SEED = 42

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)


def savefig(name: str):
    """Save figure to both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Load parquet files

Reload the per-DDF parquet files produced by notebook `01c`.  
No API call is made — we work entirely from cached data.

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DEEP_FIELDS:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found at {pq} — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue
    df = pd.read_parquet(pq)
    # Ensure numeric types for position and boolean for dipole flag
    for col in ("r:ra", "r:dec"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )
    ddf_alerts[field_name] = df
    n_tot = len(df)
    n_dip = int(df["r:isDipole"].fillna(False).sum()) if "r:isDipole" in df.columns else 0
    n_ndip = n_tot - n_dip
    print(f"[{field_name:12s}] {n_tot:6d} alerts  |  {n_dip:5d} dipoles  |  {n_ndip:5d} non-dipoles")

print("\nLoad complete.")

## 3. Angular correlation function: core utilities

We implement two backends:

- **TreeCorr backend** (preferred): handles spherical geometry, Landy–Szalay estimator,
  jackknife error bars, all efficiently.
- **NumPy/KDTree fallback**: flat-sky approximation (valid for cones ≤ 1°), slower
  but requires only standard libraries.

Both return a dict with keys `theta` (degrees), `w`, `w_err`.


In [ ]:
def make_random_catalogue(
    ra_center: float,
    dec_center: float,
    radius_deg: float,
    n: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Generate *n* uniformly-distributed random points inside a circular cone
    of radius *radius_deg* centred on (ra_center, dec_center).

    Uses the flat-sky approximation (valid for radius <= ~2 deg):
      dx ~ Uniform(-r, r)  (RA offset in degrees)
      dy ~ Uniform(-r, r)  (Dec offset in degrees)
      kept if sqrt(dx^2 + dy^2) <= r

    Returns
    -------
    ra_rand, dec_rand : arrays of shape (n,) in degrees
    """
    ra_rand = np.empty(n)
    dec_rand = np.empty(n)
    cos_dec = np.cos(np.radians(dec_center))  # RA compression factor
    filled = 0
    while filled < n:
        needed = (n - filled) * 4 // 3 + 100  # over-sample to avoid many loops
        dx = rng.uniform(-radius_deg, radius_deg, needed) / cos_dec
        dy = rng.uniform(-radius_deg, radius_deg, needed)
        # Angular distance in flat sky: use cos_dec-corrected RA
        dist = np.sqrt((dx * cos_dec) ** 2 + dy**2)
        mask = dist <= radius_deg
        take = min(mask.sum(), n - filled)
        ra_rand[filled : filled + take] = ra_center + dx[mask][:take]
        dec_rand[filled : filled + take] = dec_center + dy[mask][:take]
        filled += take
    return ra_rand, dec_rand


print("make_random_catalogue() defined.")

In [ ]:
# ── Log-spaced angular bins (shared by both backends) ─────────────────────────
THETA_EDGES = np.logspace(
    np.log10(THETA_MIN_DEG),
    np.log10(THETA_MAX_DEG),
    N_THETA_BINS + 1,
)
THETA_CENTERS = np.sqrt(THETA_EDGES[:-1] * THETA_EDGES[1:])  # geometric mean

print(f"Angular bins: {N_THETA_BINS}  from {THETA_MIN_DEG:.4f} to {THETA_MAX_DEG:.3f} deg")
print(f"Bin edges (deg): {THETA_EDGES}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TreeCorr backend
# ─────────────────────────────────────────────────────────────────────────────


def _acf_treecorr_bad(
    ra1: np.ndarray,
    dec1: np.ndarray,
    ra2: np.ndarray | None,
    dec2: np.ndarray | None,
    ra_rand: np.ndarray,
    dec_rand: np.ndarray,
    theta_edges: np.ndarray,
    is_cross: bool = False,
) -> dict:
    """
    Compute the Landy-Szalay angular (cross-)correlation function using TreeCorr.

    For auto-correlation : ra2/dec2 must be None  (uses ra1/dec1 as both D).
    For cross-correlation: ra2/dec2 is the second data catalogue.

    Returns dict with keys: theta, w, w_err (all arrays of length N_THETA_BINS).
    """
    min_sep = float(theta_edges[0]) * 60.0  # arcmin
    max_sep = float(theta_edges[-1]) * 60.0  # arcmin
    nbins = len(theta_edges) - 1

    bin_slop = 0.1  # tolerance: small = more accurate but slower

    cat1 = treecorr.Catalog(ra=ra1, dec=dec1, ra_units="degrees", dec_units="degrees")
    catr = treecorr.Catalog(ra=ra_rand, dec=dec_rand, ra_units="degrees", dec_units="degrees")

    kw = dict(
        min_sep=min_sep,
        max_sep=max_sep,
        nbins=nbins,
        sep_units="arcmin",
        bin_slop=bin_slop,
    )

    if not is_cross:
        # Auto-correlation: DD, DR, RR
        dd = treecorr.NNCorrelation(**kw)
        dr = treecorr.NNCorrelation(**kw)
        rr = treecorr.NNCorrelation(**kw)
        dd.process(cat1)
        dr.process(cat1, catr)
        rr.process(catr)
        w, w_err, _ = dd.calculateXi(rr=rr, dr=dr)
        theta_arcmin = np.exp(dd.meanlogr)
    else:
        # Cross-correlation: D1D2, D1R, D2R, RR
        cat2 = treecorr.Catalog(ra=ra2, dec=dec2, ra_units="degrees", dec_units="degrees")
        d1d2 = treecorr.NNCorrelation(**kw)
        d1r = treecorr.NNCorrelation(**kw)
        d2r = treecorr.NNCorrelation(**kw)
        rr = treecorr.NNCorrelation(**kw)
        d1d2.process(cat1, cat2)
        d1r.process(cat1, catr)
        d2r.process(cat2, catr)
        rr.process(catr)
        w, w_err, _ = d1d2.calculateXi(rr=rr, dr=d1r, rd=d2r)
        theta_arcmin = np.exp(d1d2.meanlogr)

    return {
        "theta": theta_arcmin / 60.0,  # back to degrees
        "w": w,
        "w_err": w_err,
    }


print("_acf_treecorr_bad() defined.")


def _acf_treecorr(
    ra1,
    dec1,
    ra2,
    dec2,
    ra_rand,
    dec_rand,
    theta_edges,
    is_cross=False,
):
    min_sep = float(theta_edges[0]) * 60.0
    max_sep = float(theta_edges[-1]) * 60.0
    nbins = len(theta_edges) - 1
    bin_slop = 0.1

    cat1 = treecorr.Catalog(ra=ra1, dec=dec1, ra_units="degrees", dec_units="degrees")
    catr = treecorr.Catalog(ra=ra_rand, dec=dec_rand, ra_units="degrees", dec_units="degrees")

    kw = dict(
        min_sep=min_sep,
        max_sep=max_sep,
        nbins=nbins,
        sep_units="arcmin",
        bin_slop=bin_slop,
    )

    if not is_cross:
        dd = treecorr.NNCorrelation(**kw)
        dr = treecorr.NNCorrelation(**kw)
        rr = treecorr.NNCorrelation(**kw)
        dd.process(cat1)
        dr.process(cat1, catr)
        rr.process(catr)
        # calculateXi returns (xi, varxi) in recent treecorr versions
        xi_result = dd.calculateXi(rr=rr, dr=dr)
        w = xi_result[0]
        w_err = np.sqrt(np.abs(xi_result[1]))  # varxi → sigma
        theta_arcmin = np.exp(dd.meanlogr)
    else:
        cat2 = treecorr.Catalog(ra=ra2, dec=dec2, ra_units="degrees", dec_units="degrees")
        d1d2 = treecorr.NNCorrelation(**kw)
        d1r = treecorr.NNCorrelation(**kw)
        d2r = treecorr.NNCorrelation(**kw)
        rr = treecorr.NNCorrelation(**kw)
        d1d2.process(cat1, cat2)
        d1r.process(cat1, catr)
        d2r.process(cat2, catr)
        rr.process(catr)
        xi_result = d1d2.calculateXi(rr=rr, dr=d1r, rd=d2r)
        w = xi_result[0]
        w_err = np.sqrt(np.abs(xi_result[1]))
        theta_arcmin = np.exp(d1d2.meanlogr)

    return {
        "theta": theta_arcmin / 60.0,
        "w": w,
        "w_err": w_err,
    }


print(
    f"treecorr version: {treecorr.__version__}  — calculateXi returns {len(treecorr.NNCorrelation.__doc__ or '')} chars doc"
)
print("_acf_treecorr() redefined with version-safe unpacking.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NumPy / KD-tree fallback (flat-sky Landy-Szalay)
# ─────────────────────────────────────────────────────────────────────────────


def _count_pairs_kdtree(
    ra1: np.ndarray,
    dec1: np.ndarray,
    ra2: np.ndarray,
    dec2: np.ndarray,
    theta_edges: np.ndarray,
    cos_dec: float,
) -> np.ndarray:
    """
    Count pairs (ra1, dec1) × (ra2, dec2) in each angular bin.

    Uses flat-sky coordinates:
        x = ra  * cos(dec_center)
        y = dec
    for an angular distance metric in degrees.

    Returns
    -------
    counts : array of shape (len(theta_edges)-1,)  — pair counts per bin
    """
    # Project to flat-sky (x in RA direction, corrected for cos(dec))
    xy1 = np.column_stack([ra1 * cos_dec, dec1])
    xy2 = np.column_stack([ra2 * cos_dec, dec2])

    tree2 = KDTree(xy2)

    counts = np.zeros(len(theta_edges) - 1, dtype=np.float64)
    # Query cumulative counts at each bin edge, then difference
    cum = np.zeros(len(theta_edges), dtype=np.float64)
    for k, r in enumerate(theta_edges):
        # Number of pairs within radius r
        hits = tree2.query_ball_point(xy1, r, workers=-1)
        cum[k] = sum(len(h) for h in hits)
    counts = np.diff(cum)
    return counts


def _acf_numpy(
    ra1: np.ndarray,
    dec1: np.ndarray,
    ra2: np.ndarray | None,
    dec2: np.ndarray | None,
    ra_rand: np.ndarray,
    dec_rand: np.ndarray,
    theta_edges: np.ndarray,
    dec_center: float,
    is_cross: bool = False,
) -> dict:
    """
    Flat-sky Landy-Szalay estimator using a KD-tree for pair counting.

    Poisson error: w_err = (1 + w) / sqrt(DD) where DD is the raw pair count.
    Returns dict with keys: theta (deg), w, w_err.
    """
    cos_dec = np.cos(np.radians(dec_center))
    n1 = len(ra1)
    nr = len(ra_rand)
    # Normalisation factors (number of distinct pairs)
    f_dd = 1.0 / (n1 * (n1 - 1))
    f_rr = 1.0 / (nr * (nr - 1))
    f_dr = 1.0 / (n1 * nr)

    if not is_cross:
        DD = _count_pairs_kdtree(ra1, dec1, ra1, dec1, theta_edges, cos_dec)
        DR = _count_pairs_kdtree(ra1, dec1, ra_rand, dec_rand, theta_edges, cos_dec)
        RR = _count_pairs_kdtree(ra_rand, dec_rand, ra_rand, dec_rand, theta_edges, cos_dec)
        # Remove self-pairs from auto-counts
        # (KD-tree counts each pair twice + self; correct accordingly)
        DD_norm = DD * f_dd
        DR_norm = DR * f_dr
        RR_norm = RR * f_rr
    else:
        n2 = len(ra2)
        f_d1d2 = 1.0 / (n1 * n2)
        f_d1r = 1.0 / (n1 * nr)
        f_d2r = 1.0 / (n2 * nr)
        D1D2 = _count_pairs_kdtree(ra1, dec1, ra2, dec2, theta_edges, cos_dec)
        D1R = _count_pairs_kdtree(ra1, dec1, ra_rand, dec_rand, theta_edges, cos_dec)
        D2R = _count_pairs_kdtree(ra2, dec2, ra_rand, dec_rand, theta_edges, cos_dec)
        RR = _count_pairs_kdtree(ra_rand, dec_rand, ra_rand, dec_rand, theta_edges, cos_dec)
        DD_norm = D1D2 * f_d1d2
        DR_norm = 0.5 * (D1R * f_d1r + D2R * f_d2r)
        RR_norm = RR * f_rr

    with np.errstate(invalid="ignore", divide="ignore"):
        w = np.where(RR_norm > 0, (DD_norm - 2.0 * DR_norm + RR_norm) / RR_norm, np.nan)
        # Poisson-like error estimate
        raw_dd = DD if not is_cross else D1D2
        w_err = np.where(raw_dd > 0, (1.0 + w) / np.sqrt(raw_dd + 1e-30), np.nan)

    theta_centers = np.sqrt(theta_edges[:-1] * theta_edges[1:])

    return {
        "theta": theta_centers,
        "w": w,
        "w_err": w_err,
    }


print("_acf_numpy() defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Public wrapper: dispatches to TreeCorr or NumPy
# ─────────────────────────────────────────────────────────────────────────────


def angular_correlation(
    ra1: np.ndarray,
    dec1: np.ndarray,
    ra_center: float,
    dec_center: float,
    ra2: np.ndarray | None = None,
    dec2: np.ndarray | None = None,
    n_random: int = N_RANDOM,
    seed: int = RANDOM_SEED,
) -> dict | None:
    """
    Compute the Landy-Szalay angular (auto or cross) correlation function.

    Parameters
    ----------
    ra1, dec1    : positions of data catalogue 1 (degrees)
    ra_center,
    dec_center   : DDF field centre — used to generate the random catalogue
    ra2, dec2    : positions of data catalogue 2 (cross-correlation only;
                   None for auto-correlation)
    n_random     : size of the random catalogue
    seed         : random seed

    Returns
    -------
    dict with keys theta (deg), w, w_err — or None if not enough points.
    """
    is_cross = (ra2 is not None) and (dec2 is not None)
    n_min = 20  # minimum number of points to attempt the computation

    if len(ra1) < n_min:
        print(f"  Not enough points in catalogue 1 (n={len(ra1)} < {n_min}) — skipping.")
        return None
    if is_cross and len(ra2) < n_min:
        print(f"  Not enough points in catalogue 2 (n={len(ra2)} < {n_min}) — skipping.")
        return None

    rng = np.random.default_rng(seed)
    ra_rand, dec_rand = make_random_catalogue(ra_center, dec_center, CONE_RADIUS_DEG, n_random, rng)

    if HAS_TREECORR:
        return _acf_treecorr(
            ra1,
            dec1,
            ra2,
            dec2,
            ra_rand,
            dec_rand,
            THETA_EDGES,
            is_cross=is_cross,
        )
    else:
        return _acf_numpy(
            ra1,
            dec1,
            ra2,
            dec2,
            ra_rand,
            dec_rand,
            THETA_EDGES,
            dec_center=dec_center,
            is_cross=is_cross,
        )


print("angular_correlation() dispatcher defined.")

## 4. Utility: plot $w(\theta)$ with zero-line and error bars

In [ ]:
def plot_wcf(
    ax: plt.Axes,
    result: dict,
    label: str = "",
    color: str = "steelblue",
    marker: str = "o",
) -> None:
    """
    Plot w(theta) with error bars on a given Axes.

    The zero-line (uniform Poisson distribution) is drawn as a dashed grey line.
    """
    if result is None:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        return

    theta = result["theta"]
    w = result["w"]
    w_err = result["w_err"]

    # Replace NaN errors with 0 for display purposes
    w_err_plot = np.where(np.isfinite(w_err), w_err, 0.0)

    ax.axhline(0.0, color="grey", lw=0.8, ls="--", zorder=0, label="w=0 (uniform)")
    ax.errorbar(
        theta * 60,  # convert degrees → arcmin for legibility
        w,
        yerr=w_err_plot,
        fmt=marker + "-",
        color=color,
        ms=5,
        lw=1.2,
        capsize=3,
        label=label,
    )
    ax.set_xscale("log")
    ax.set_xlabel(r"$\theta$ (arcmin)")
    ax.set_ylabel(r"$w(\theta)$")


print("plot_wcf() defined.")

## 5. Section A — Auto-correlation of **all alerts** per DDF

**Goal**: check whether the full alert catalogue is spatially uniform,
or whether there are angular clustering features at scales < 1 degree
(e.g. CCD gaps, bad columns, bright-star masks).

A flat $w(\theta) \approx 0$ indicates uniform/uncorrelated detections.


In [ ]:
# ── Compute auto-correlation for all alerts in each DDF ───────────────────────
acf_all: dict[str, dict | None] = {}

for field_name, (ra_c, dec_c) in DEEP_FIELDS.items():
    df = ddf_alerts.get(field_name, pd.DataFrame())
    if df.empty or "r:ra" not in df.columns:
        acf_all[field_name] = None
        print(f"[{field_name:12s}] no data — skipped.")
        continue

    sub = df[["r:ra", "r:dec"]].dropna()
    print(f"[{field_name:12s}] computing ACF for {len(sub):,} alerts …", end=" ", flush=True)
    result = angular_correlation(
        sub["r:ra"].values,
        sub["r:dec"].values,
        ra_center=ra_c,
        dec_center=dec_c,
    )
    acf_all[field_name] = result
    if result is not None:
        print("done.")

print("\nSection A complete.")

In [ ]:
# ── Plot ACF(all) for each DDF ────────────────────────────────────────────────
fields_with_data = [(fn, r) for fn, r in acf_all.items() if r is not None]
n_fields = len(fields_with_data)

if n_fields == 0:
    print("No ACF results to plot.")
else:
    ncols = min(3, n_fields)
    nrows = int(np.ceil(n_fields / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
    colors = plt.cm.tab10(np.linspace(0, 1, n_fields))

    for idx, (field_name, result) in enumerate(fields_with_data):
        ax = axes[idx // ncols][idx % ncols]
        plot_wcf(ax, result, label=f"{field_name} (all)", color=colors[idx])
        n_alerts = len(ddf_alerts[field_name].dropna(subset=["r:ra"]))
        ax.set_title(f"{field_name}  (N={n_alerts:,})", fontsize=9)
        ax.legend(fontsize=7)

    # Hide unused axes
    for idx in range(n_fields, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(r"Auto-correlation $w(\theta)$ — all DIA alerts per DDF", fontsize=11, y=1.01)
    plt.tight_layout()
    savefig("acf_all_alerts_per_ddf")
    plt.show()

## 6. Section B — Auto-correlation of **dipole alerts** per DDF

**Goal**: test whether dipole-flagged DIA sources cluster angularly.

If dipoles are caused by template mis-registration, they may cluster near
bright stars (PSF wings), saturated columns, or on specific detectors.
A significant positive $w(\theta)$ at small scales (<< 1 arcmin) would support this.


In [ ]:
# ── Compute auto-correlation for dipole-only alerts ───────────────────────────
acf_dip: dict[str, dict | None] = {}

for field_name, (ra_c, dec_c) in DEEP_FIELDS.items():
    df = ddf_alerts.get(field_name, pd.DataFrame())
    if df.empty or "r:isDipole" not in df.columns:
        acf_dip[field_name] = None
        print(f"[{field_name:12s}] no data — skipped.")
        continue

    mask_dip = df["r:isDipole"].fillna(False).astype(bool)
    sub = df.loc[mask_dip, ["r:ra", "r:dec"]].dropna()

    print(f"[{field_name:12s}] computing ACF for {len(sub):,} dipole alerts …", end=" ", flush=True)
    result = angular_correlation(
        sub["r:ra"].values,
        sub["r:dec"].values,
        ra_center=ra_c,
        dec_center=dec_c,
    )
    acf_dip[field_name] = result
    if result is not None:
        print("done.")

print("\nSection B complete.")

In [ ]:
# ── Plot ACF(dipoles) ─────────────────────────────────────────────────────────
fields_dip_data = [(fn, r) for fn, r in acf_dip.items() if r is not None]
n_dip_fields = len(fields_dip_data)

if n_dip_fields == 0:
    print("No dipole ACF results to plot.")
else:
    ncols = min(3, n_dip_fields)
    nrows = int(np.ceil(n_dip_fields / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
    colors = plt.cm.Reds(np.linspace(0.4, 0.9, n_dip_fields))

    for idx, (field_name, result) in enumerate(fields_dip_data):
        ax = axes[idx // ncols][idx % ncols]
        df = ddf_alerts[field_name]
        n_d = int(df["r:isDipole"].fillna(False).astype(bool).sum())
        plot_wcf(ax, result, label=f"{field_name} (dipoles)", color=colors[idx], marker="s")
        ax.set_title(f"{field_name}  N_dipoles={n_d:,}", fontsize=9)
        ax.legend(fontsize=7)

    for idx in range(n_dip_fields, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(r"Auto-correlation $w(\theta)$ — dipole alerts per DDF", fontsize=11, y=1.01)
    plt.tight_layout()
    savefig("acf_dipoles_per_ddf")
    plt.show()

## 7. Section C — Cross-correlation non-dipoles × dipoles per DDF

**Goal**: compare the spatial distribution of dipole alerts relative to
non-dipole alerts.

If dipoles trace the same underlying sky density as non-dipoles, the
cross-correlation should match the auto-correlation of all alerts.
A significant **positive** $w_{cross}(\theta)$ at small scales means dipoles
appear preferentially close to non-dipole sources (same cluster).
A **negative** cross-correlation would indicate spatial avoidance — dipoles
in regions where normal alerts are rare (e.g. bad-template zones).


In [ ]:
# ── Compute cross-correlation non-dipole × dipole ─────────────────────────────
xcf_nodip_dip: dict[str, dict | None] = {}

for field_name, (ra_c, dec_c) in DEEP_FIELDS.items():
    df = ddf_alerts.get(field_name, pd.DataFrame())
    if df.empty or "r:isDipole" not in df.columns:
        xcf_nodip_dip[field_name] = None
        print(f"[{field_name:12s}] no data — skipped.")
        continue

    mask_dip = df["r:isDipole"].fillna(False).astype(bool)
    mask_ndip = ~mask_dip

    sub_dip = df.loc[mask_dip, ["r:ra", "r:dec"]].dropna()
    sub_ndip = df.loc[mask_ndip, ["r:ra", "r:dec"]].dropna()

    n_dip = len(sub_dip)
    n_ndip = len(sub_ndip)
    print(
        f"[{field_name:12s}] cross-corr  non-dipoles({n_ndip:,}) × dipoles({n_dip:,}) …",
        end=" ",
        flush=True,
    )

    result = angular_correlation(
        sub_ndip["r:ra"].values,
        sub_ndip["r:dec"].values,
        ra_center=ra_c,
        dec_center=dec_c,
        ra2=sub_dip["r:ra"].values,
        dec2=sub_dip["r:dec"].values,
    )
    xcf_nodip_dip[field_name] = result
    if result is not None:
        print("done.")

print("\nSection C complete.")

In [ ]:
# ── Plot cross-correlation per DDF ────────────────────────────────────────────
fields_xcf = [(fn, r) for fn, r in xcf_nodip_dip.items() if r is not None]
n_xcf = len(fields_xcf)

if n_xcf == 0:
    print("No cross-correlation results to plot.")
else:
    ncols = min(3, n_xcf)
    nrows = int(np.ceil(n_xcf / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
    colors = plt.cm.Greens(np.linspace(0.4, 0.9, n_xcf))

    for idx, (field_name, result) in enumerate(fields_xcf):
        ax = axes[idx // ncols][idx % ncols]
        plot_wcf(ax, result, label=f"{field_name} (non-dip × dip)", color=colors[idx], marker="^")
        ax.set_title(field_name, fontsize=9)
        ax.legend(fontsize=7)

    for idx in range(n_xcf, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(r"Cross-correlation $w(\theta)$ — non-dipoles $\times$ dipoles per DDF", fontsize=11, y=1.01)
    plt.tight_layout()
    savefig("xcf_nodip_dip_per_ddf")
    plt.show()

## 8. Comparison plot: ACF(all) vs ACF(dipoles) vs XCF

Overlay all three $w(\theta)$ on the same axes for each DDF,
allowing direct visual comparison of the spatial statistics.


In [ ]:
all_fields = list(DEEP_FIELDS.keys())
n_fields_total = len(all_fields)
ncols = min(3, n_fields_total)
nrows = int(np.ceil(n_fields_total / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

for idx, field_name in enumerate(all_fields):
    ax = axes[idx // ncols][idx % ncols]
    ax.axhline(0.0, color="grey", lw=0.8, ls="--", zorder=0)

    r_all = acf_all.get(field_name)
    r_dip = acf_dip.get(field_name)
    r_xcf = xcf_nodip_dip.get(field_name)

    if r_all is not None:
        w_e = np.where(np.isfinite(r_all["w_err"]), r_all["w_err"], 0.0)
        ax.errorbar(
            r_all["theta"] * 60,
            r_all["w"],
            yerr=w_e,
            fmt="o-",
            ms=4,
            lw=1.2,
            capsize=2,
            color="steelblue",
            label="all alerts",
        )

    if r_dip is not None:
        w_e = np.where(np.isfinite(r_dip["w_err"]), r_dip["w_err"], 0.0)
        ax.errorbar(
            r_dip["theta"] * 60,
            r_dip["w"],
            yerr=w_e,
            fmt="s--",
            ms=4,
            lw=1.2,
            capsize=2,
            color="crimson",
            label="dipoles only",
        )

    if r_xcf is not None:
        w_e = np.where(np.isfinite(r_xcf["w_err"]), r_xcf["w_err"], 0.0)
        ax.errorbar(
            r_xcf["theta"] * 60,
            r_xcf["w"],
            yerr=w_e,
            fmt="^:",
            ms=4,
            lw=1.2,
            capsize=2,
            color="seagreen",
            label="non-dip × dip",
        )

    ax.set_xscale("log")
    ax.set_xlabel(r"$\theta$ (arcmin)")
    ax.set_ylabel(r"$w(\theta)$")
    ax.set_title(field_name, fontsize=9)
    ax.legend(fontsize=7, loc="upper right")

for idx in range(n_fields_total, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(r"Angular correlation functions per DDF: all / dipoles / cross", fontsize=11, y=1.01)
plt.tight_layout()
savefig("wcf_comparison_all_dip_xcf_per_ddf")
plt.show()

## 8.c Summary for comparison of correlation functions in DDF

- La bande grise autour de zéro (axhspan) aide à voir quelles barres d'erreur touchent le niveau de Poisson.
- Les légendes affichent le nombre total d'alertes (ou de dipoles) pour chaque DDF — utile pour interpréter le niveau de bruit.
- Les 3 figures sont sauvegardées sous SUMMARY_acf_all_alerts, SUMMARY_acf_dipoles, SUMMARY_xcf_nodip_dip dans figs_DIPOLES_02/.

In [ ]:
# ── Summary plots: one panel per correlation type, all DDFs overlaid ──────────

# Colour palette: maximally distinct colours for up to 7 DDFs
FIELD_COLORS = {
    "COSMOS": "#e6194b",  # red
    "ELAIS-S1": "#3cb44b",  # green
    "ECDFS": "#4363d8",  # blue
    "EDFS-a": "#f58231",  # orange
    "EDFS-b": "#911eb4",  # purple
    "EDFS": "#42d4f4",  # cyan
    "M49": "#f032e6",  # magenta
}
FIELD_MARKERS = {
    "COSMOS": "o",
    "ELAIS-S1": "s",
    "ECDFS": "^",
    "EDFS-a": "D",
    "EDFS-b": "v",
    "EDFS": "P",
    "M49": "X",
}


def plot_wcf_summary(
    results_dict: dict,
    title: str,
    figname: str,
    ylabel: str = r"$w(\theta)$",
) -> None:
    """
    Overlay w(theta) curves for all DDFs on a single panel.

    Parameters
    ----------
    results_dict : dict  field_name → result dict (theta, w, w_err) or None
    title        : figure suptitle
    figname      : filename stem for savefig
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axhline(0.0, color="grey", lw=0.9, ls="--", zorder=0, label="_nolegend_")

    plotted = 0
    for field_name, result in results_dict.items():
        if result is None:
            continue
        theta = result["theta"] * 60.0  # degrees → arcmin
        w = np.asarray(result["w"], dtype=float)
        w_err = np.asarray(result["w_err"], dtype=float)
        w_err = np.where(np.isfinite(w_err), w_err, 0.0)

        color = FIELD_COLORS.get(field_name, "black")
        marker = FIELD_MARKERS.get(field_name, "o")

        # Count alerts for the legend label
        df = ddf_alerts.get(field_name, pd.DataFrame())
        n = len(df.dropna(subset=["r:ra"])) if not df.empty else 0

        ax.errorbar(
            theta,
            w,
            yerr=w_err,
            fmt=f"{marker}-",
            color=color,
            ms=6,
            lw=1.5,
            capsize=3,
            capthick=1.2,
            elinewidth=0.9,
            label=f"{field_name}  (N={n:,})",
            zorder=3,
        )
        plotted += 1

    if plotted == 0:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)

    ax.set_xscale("log")
    ax.set_xlabel(r"$\theta$ (arcmin)", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(
        loc="upper right",
        fontsize=8,
        framealpha=0.9,
        edgecolor="grey",
        ncol=1,
    )
    # Light horizontal band around zero for readability
    ax.axhspan(-0.05, 0.05, color="grey", alpha=0.07, zorder=0)

    plt.tight_layout()
    savefig(figname)
    plt.show()


# ── (A) All alerts ─────────────────────────────────────────────────────────────
plot_wcf_summary(
    acf_all,
    title=r"Auto-correlation $w(\theta)$ — all DIA alerts — all DDFs",
    figname="SUMMARY_acf_all_alerts",
)


# ── (B) Dipoles only ──────────────────────────────────────────────────────────
# Build a version of the legend with dipole counts
def _make_dip_results_with_label():
    """Return results dict with dipole-count annotation baked in."""
    return acf_dip  # labels computed inside plot_wcf_summary from ddf_alerts


plot_wcf_summary(
    acf_dip,
    title=r"Auto-correlation $w(\theta)$ — dipole alerts only — all DDFs",
    figname="SUMMARY_acf_dipoles",
)

# ── (C) Cross-correlation non-dipoles × dipoles ───────────────────────────────
plot_wcf_summary(
    xcf_nodip_dip,
    title=r"Cross-correlation $w(\theta)$ — non-dipoles $\times$ dipoles — all DDFs",
    figname="SUMMARY_xcf_nodip_dip",
)

## 9. Save correlation results to parquet

Serialise all $w(\theta)$ results to `data_DIPOLES_02/` for downstream analysis.

In [ ]:
def save_wcf_result(
    result: dict | None,
    field_name: str,
    label: str,  # e.g. 'all', 'dipoles', 'cross_nodip_dip'
) -> None:
    """Save a single w(theta) result as a parquet file."""
    if result is None:
        return
    df_out = pd.DataFrame(
        {
            "theta_deg": result["theta"],
            "theta_arcmin": result["theta"] * 60.0,
            "w": result["w"],
            "w_err": result["w_err"],
            "field": field_name,
            "label": label,
        }
    )
    safe = field_name.replace("-", "_").replace(" ", "_")
    path = os.path.join(DIR_DATA, f"wcf_{label}_{safe}.parquet")
    df_out.to_parquet(path, index=False)
    print(f"  saved {path}")


print("Saving correlation results …")
for field_name in DEEP_FIELDS:
    save_wcf_result(acf_all.get(field_name), field_name, "all")
    save_wcf_result(acf_dip.get(field_name), field_name, "dipoles")
    save_wcf_result(xcf_nodip_dip.get(field_name), field_name, "cross_nodip_dip")

print("Done.")

## 10. Quick uniformity summary

For each DDF and catalogue type we compute a simple χ² against the null hypothesis
$w(\theta) = 0$ (uniform distribution) as a scalar figure of merit.

$$\chi^2_{\rm uniform} = \sum_i \left( \frac{w_i}{\sigma_i} \right)^2$$

A large value indicates significant departure from uniformity.
This is purely indicative — no formal $p$-value correction for multiple bins is applied here.

In [ ]:
def chi2_vs_zero(result: dict | None) -> float:
    """Return reduced chi2 of w(theta) relative to the w=0 hypothesis."""
    if result is None:
        return np.nan
    w = np.asarray(result["w"], dtype=float)
    err = np.asarray(result["w_err"], dtype=float)
    good = np.isfinite(w) & np.isfinite(err) & (err > 0)
    if good.sum() == 0:
        return np.nan
    chi2 = np.sum((w[good] / err[good]) ** 2)
    return float(chi2 / good.sum())  # reduced chi2


rows = []
for field_name in DEEP_FIELDS:
    rows.append(
        {
            "field": field_name,
            "chi2_all": chi2_vs_zero(acf_all.get(field_name)),
            "chi2_dipoles": chi2_vs_zero(acf_dip.get(field_name)),
            "chi2_cross": chi2_vs_zero(xcf_nodip_dip.get(field_name)),
        }
    )

df_chi2 = pd.DataFrame(rows).set_index("field")
print("Reduced chi2 vs w=0 per DDF:")
print(df_chi2.to_string(float_format="{:.2f}".format))
print("\nchi2 >> 1  →  significant clustering / non-uniformity")
print("chi2 ~ 1   →  consistent with uniform distribution")

df_chi2.to_csv(os.path.join(DIR_DATA, "chi2_uniformity_summary.csv"))
print(f"\nSaved chi2 summary to {os.path.join(DIR_DATA, 'chi2_uniformity_summary.csv')}")

## 11. Discussion and caveats

### Interpretation

- **Positive $w(\theta)$ at small scales** (< 1 arcmin): suggests clustering tighter
  than Poisson — likely real astrophysical sources (galaxy clusters, multi-epoch
  detection of the same object) or detector artefacts (bad columns, cosmic rays).
- **Positive $w(\theta)$ at large scales** (> 10 arcmin): suggests the field footprint
  is non-uniform (survey strategy, variable depth, CCD gaps).
- **Dipole ACF >> all-alert ACF**: dipoles are more clustered than the general population
  → spatially localised systematics.
- **Positive cross-correlation**: dipoles tend to appear where there are also many
  non-dipole detections → same dense regions, not a separate population.

### Caveats and limitations

1. **Flat-sky approximation**: valid for our 1° cones, but small errors at the edges.
2. **No survey mask**: we use a simple circular random catalogue, not the true
   LSST/Rubin focal-plane footprint.  The masked CCD gaps will appear as
   negative $w(\theta)$ at the gap scale (~10 arcmin).  A proper mask
   (from the Butler or a `healsparse` map) would be needed for a rigorous test.
3. **Finite-size effects**: with small samples ($N < 200$) the Poisson noise
   dominates and $w(\theta)$ is very noisy.
4. **Treecorr vs NumPy**: the NumPy fallback uses a flat-sky KD-tree and
   self-pair correction that may differ slightly from TreeCorr's spherical result.

### Next steps

- Use the Butler focal-plane mask to build a proper random catalogue  
  (notebook `03_dipoles_proper_mask.ipynb`).
- Check whether dipole clustering is correlated with CCD number (`r:detector`)
  or pixel position (`r:x`, `r:y`).
- Compute the projected correlation function $w_p(r_p)$ in physical units
  once photo-z estimates are available.
